In [30]:
import re
import numpy as np


def load_file_content(filepath):
    try:
        with open(filepath, "r") as f:
            return f.read()
    except FileNotFoundError:
        return None
    
def extract_die_area(def_text):
    for line in def_text.splitlines():
        if line.startswith("DIEAREA"):
            # Extract numbers inside parentheses
            # This finds all sequences of digits
            nums = re.findall(r'\d+', line)
            if len(nums) == 4:
                die_x_min = int(nums[0])
                die_y_min = int(nums[1])
                die_x_max = int(nums[2])
                die_y_max = int(nums[3])
                return die_x_min, die_y_min, die_x_max, die_y_max
    return RuntimeError("DIEAREA not found in DEF file")



In [31]:
#extra def text
design_path = "./CTS-Bench/runs/aes_run_20260302_201328/11-openroad-detailedplacement/aes.def"
saif_path = "./CTS-Bench/runs/aes_run_20260302_201328/aes.saif"
def_text = load_file_content(design_path)
saif_text = load_file_content(saif_path)

In [34]:
import re

# 1. Global Registry (Session-wide)
if 'GLOBAL_CELL_VOCAB' not in globals():
    GLOBAL_CELL_VOCAB = {}
    NUM_CELL_TYPES = 0

def build_cell_vocabulary(lib_path="sky130_fd_sc_hd_tt_025C_1v80.lib"):
    """
    Builds a deterministic mapping of every cell in the PDK to a unique ID.
    Safe for 'Run All' in Jupyter.
    """
    global GLOBAL_CELL_VOCAB, NUM_CELL_TYPES
    
    # If already built, don't re-parse the massive file
    if GLOBAL_CELL_VOCAB:
        return GLOBAL_CELL_VOCAB

    print(f"Initializing Universal Cell Vocabulary from {lib_path}...")
    lib_text = load_file_content(lib_path)
    
    if lib_text is None:
        print(f"Error: {lib_path} not found. Ensure it's in the current directory.")
        return {}

    # Extract all cell names and sort for deterministic IDs
    cell_names = re.findall(r'cell\s*\("([^"]+)"\)', lib_text)
    unique_sorted_cells = sorted(set(cell_names))
    
    GLOBAL_CELL_VOCAB = {name: i for i, name in enumerate(unique_sorted_cells)}
    NUM_CELL_TYPES = len(GLOBAL_CELL_VOCAB)
    
    print(f"Vocabulary complete: {NUM_CELL_TYPES} unique Sky130 cells registered.")
    return GLOBAL_CELL_VOCAB

# Initialize it immediately
GLOBAL_CELL_VOCAB = build_cell_vocabulary()

Initializing Universal Cell Vocabulary from sky130_fd_sc_hd_tt_025C_1v80.lib...
Vocabulary complete: 428 unique Sky130 cells registered.


In [35]:
import re

import re

_lib_content_cache = {}
_cell_data_cache = {}

def get_cell_electrical_and_physical_data(cell_name, lib_path="sky130_fd_sc_hd_tt_025C_1v80.lib"):
    global _cell_data_cache, _lib_content_cache
    
    if cell_name in _cell_data_cache:
        return _cell_data_cache[cell_name]
        
    if "text" not in _lib_content_cache:
        print("Loading LIB file into memory...")
        _lib_content_cache["text"] = load_file_content(lib_path)
        
    lib_text = _lib_content_cache["text"]
    
    cell_marker = f'cell ("{cell_name}") {{'
    start_idx = lib_text.find(cell_marker)
    
    if start_idx == -1:
        print(f"Warning: Cell {cell_name} not found in LIB.")
        _cell_data_cache[cell_name] = {"avg_cap": 0.0, "total_cap": 0.0, "area": 0.0}
        return _cell_data_cache[cell_name]
        
    end_idx = lib_text.find('cell ("', start_idx + len(cell_marker))
    if end_idx == -1:
        end_idx = len(lib_text)
        
    cell_block = lib_text[start_idx:end_idx]
    
    # --- 1. Extract Area ---
    area_match = re.search(r'area\s*:\s*([0-9\.]+)\s*;', cell_block)
    cell_area = float(area_match.group(1)) if area_match else 0.0

    # --- 2. Extract Capacitance ---
    pin_blocks = cell_block.split('pin ("')[1:]
    input_caps = []
    
    for pin in pin_blocks:
        if 'direction : "input"' in pin or 'direction : input' in pin:
            cap_match = re.search(r'capacitance\s*:\s*([0-9\.]+)\s*;', pin)
            if cap_match:
                input_caps.append(float(cap_match.group(1)))
                
    if input_caps:
        total_cap = sum(input_caps)
        avg_cap = total_cap / len(input_caps)
    else:
        total_cap = 0.0
        avg_cap = 0.0
        
    # --- 3. Save to Cache ---
    result = {"avg_cap": avg_cap, "total_cap": total_cap, "area": cell_area}
    _cell_data_cache[cell_name] = result
    
    return result

import numpy as np
import re

import re
import numpy as np

def build_activity_cache(saif_text):
    """
    Parses the entire SAIF text using the robust parenthesis-balancing method.
    Extracts log1p(Toggle Count) and Signal Probability (SP) for all instances.
    """
    activity_cache = {}
    if not saif_text:
        return activity_cache
        
    start_pattern = re.compile(r'\(INSTANCE\s+([a-zA-Z0-9_]+)')
    tc_pattern = re.compile(r'\(TC\s+(\d+)\)')
    t1_pattern = re.compile(r'\(T1\s+(\d+)\)')
    t0_pattern = re.compile(r'\(T0\s+(\d+)\)')

    print("Building activity cache from SAIF...")
    
    for match in start_pattern.finditer(saif_text):
        gate_name = match.group(1)
        
        start_index = match.start()
        current_index = start_index
        balance = 0
        
        # --- PARENTHESIS TRACKING (Perfect for nested SAIFs) ---
        while current_index < len(saif_text):
            char = saif_text[current_index]
            if char == '(': balance += 1
            elif char == ')': balance -= 1
            if balance == 0: break
            current_index += 1
        
        full_block = saif_text[start_index : current_index + 1]
        # -------------------------------------------------------
        
        max_tc = 0
        sum_t1 = 0
        sum_t0 = 0
        sum_tc = 0
        non_zero_count = 0
        
        for line in full_block.splitlines():
            tc_match = tc_pattern.search(line)
            t1_match = t1_pattern.search(line)
            t0_match = t0_pattern.search(line)
            
            if tc_match or t1_match or t0_match:
                # Ignore CLK nets so they don't skew the logic features
                if "CLK" in line.upper() or "CLOCK" in line.upper(): 
                    continue
                    
                if tc_match:
                    val = int(tc_match.group(1))
                    if val >0 :
                        non_zero_count +=1
                    if val > max_tc: max_tc = val
                    sum_tc += val
                if t1_match:
                    sum_t1 += int(t1_match.group(1))
                if t0_match:
                    sum_t0 += int(t0_match.group(1))
        
        # Calculate the final ML features
        final_tc = np.log1p(max_tc)
        final_sum_tc = np.log1p(sum_tc)
        
        total_time = sum_t1 + sum_t0
        sp = (sum_t1 / total_time) if total_time > 0 else 0.0
        
        activity_cache[gate_name] = {
            "max_toggle_count": final_tc,
            "signal_prob": sp,
            "sum_toggle_count": final_sum_tc,
            "non_zero_count": non_zero_count
        }
            
    return activity_cache

def extract_graph_nodes(def_text, saif_text, clock_port="clk"):

    die_x_min, die_y_min, die_x_max, die_y_max = extract_die_area(def_text)
    # 1. Topologically identify all flip-flops via the clock net
    # (Using your exact logic)
    clock_pattern = rf'-\s+{re.escape(clock_port)}\s+\(\s+PIN\s+{re.escape(clock_port)}\s+\).*?;'
    clock_match = re.search(clock_pattern, def_text, re.DOTALL)
    
    if not clock_match:
        print(f"Warning: Clock net '{clock_port}' not found.")
        all_flops = set()
    else:
        # Extracts the instance names connected to the CLK pin
        all_flops = set(re.findall(r'\(\s+((?!PIN)\S+)\s+CLK\s+\)', clock_match.group(0)))
        
    print(f"Topological scan found {len(all_flops)} flip-flops connected to {clock_port}.")

    activity_cache = build_activity_cache(saif_text)

    # 2. Isolate the COMPONENTS block
    block_match = re.search(r'COMPONENTS\s+\d+\s*;(.*?)END COMPONENTS', def_text, re.DOTALL)
    components_block = block_match.group(1) if block_match else ""
    
    # 3. Define Regex Patterns for components
    line_pattern = re.compile(r'^\s*-\s+(\S+)\s+(\S+)(.*?);', re.MULTILINE)
    coord_pattern = re.compile(r'\(\s*(-?\d+)\s+(-?\d+)\s*\)')
    
    extracted_nodes = []
    ignore_keywords = ["decap", "fill", "tap", "tapvpwrvgnd", "diode", "antenna"]

    # 4. Parse components and build features
    for match in line_pattern.finditer(components_block):
        instance_name = match.group(1)
        cell_name = match.group(2)
        properties = match.group(3)
        
        if any(keyword in cell_name.lower() for keyword in ignore_keywords):
            continue
            
        coords_match = coord_pattern.search(properties)
        x = int(coords_match.group(1)) if coords_match else None
        y = int(coords_match.group(2)) if coords_match else None
            
        drive_match = re.search(r'_(\d+)$', cell_name)
        drive_strength = int(drive_match.group(1)) if drive_match else 0
            
        # TOPOLOGICAL CHECK: Is it in the flop set?
        is_sequential = 1 if instance_name in all_flops else 0
        
        is_buffer = 1 if ("buf" in cell_name.lower() or "inv" in cell_name.lower()) else 0

        cell_data = get_cell_electrical_and_physical_data(cell_name)
        cell_area = cell_data["area"]
        node_avg_cap = cell_data["avg_cap"]
        node_total_cap = cell_data["total_cap"]
        saif_data = activity_cache.get(instance_name, {"toggle_count": 0.0, "signal_prob": 0.0})

        cell_type_id = GLOBAL_CELL_VOCAB.get(cell_name, -1)
        if cell_type_id == -1:
            print(f"Warning: {cell_name} not in vocabulary")
            cell_type_id = 0  # fallback

        if x is not None and y is not None:
            dist_left = x - die_x_min
            dist_bottom = y - die_y_min
            dist_right = die_x_max - x
            dist_top = die_y_max - y
            dist_to_boundaries = [dist_left, dist_right, dist_top, dist_bottom]
        else:
            dist_to_boundaries = [0, 0, 0, 0] 
        
        extracted_nodes.append({
            "instance_name": instance_name,
            "cell_name": cell_name,
            "x": x,
            "y": y,
            "cell_area": cell_area,
            "cell_type_id": cell_type_id,
            "avg_pin_cap": node_avg_cap,
            "total_pin_cap": node_total_cap,
            "dist_to_boundaries": dist_to_boundaries,
            "toggle_count": saif_data["max_toggle_count"],
            "signal_prob": saif_data["signal_prob"],
            "sum_toggle_count": saif_data["sum_toggle_count"],
            "drive_strength": drive_strength,
            "is_sequential": is_sequential,
            "is_buffer": is_buffer, 
            "non_zero_count": saif_data["non_zero_count"]
        })

    return extracted_nodes, all_flops



# Pass saif_text into the function!
extracted_nodes, all_flops = extract_graph_nodes(def_text, saif_text=saif_text, clock_port="clk")

# Print the header for Categorical Mapping and Physical Data
print(f"Total Nodes Extracted: {len(extracted_nodes)}")
print("-" * 115)
print(f"{'Instance Name':<20} | {'Cell ID':<8} | {'Cell Name':<28} | {'Area':<8} | {'Drive':<5} | {'Seq':<3} | {'Buf':<3}")
print("-" * 115)

# Loop through the list and print each node (printing first 25 for a good sample size)
for node in extracted_nodes[:25]: 
    print(f"{node['instance_name']:<20} | {node['cell_type_id']:<8} | {node['cell_name']:<28} | {node['cell_area']:<8.2f} | {node['drive_strength']:<5} | {node['is_sequential']:<3} | {node['is_buffer']:<3}")
# Print the header
# print(f"Total Nodes Extracted: {len(extracted_nodes)}")
# print("-" * 155)
# print(f"{'Instance Name':<20} | {'Cell Name':<28} | {'(X, Y)':<18} | {'Bounds (L,R,T,B)':<28} | {'Area':<8} | {'Avg Cap':<8} | {'Tot Cap':<8} | {'Drv':<3} | {'Seq':<3} | {'Buf':<3}")
# print("-" * 155)

# # Loop through the list and print each node (printing first 20 to avoid terminal flooding)
# for node in extracted_nodes: 
#     coords = f"({node['x']}, {node['y']})"
#     bounds = str(node['dist_to_boundaries'])
    
#     # Using .2f for area and .4f for capacitance to keep the floating point numbers clean
#     print(f"{node['instance_name']:<20} | {node['cell_name']:<28} | {coords:<18} | {bounds:<28} | {node['cell_area']:<8.2f} | {node['avg_pin_cap']:<8.4f} | {node['total_pin_cap']:<8.4f} | {node['drive_strength']:<3} | {node['is_sequential']:<3} | {node['is_buffer']:<3}")



Topological scan found 2994 flip-flops connected to clk.
Building activity cache from SAIF...
Loading LIB file into memory...
Total Nodes Extracted: 24327
-------------------------------------------------------------------------------------------------------------------
Instance Name        | Cell ID  | Cell Name                    | Area     | Drive | Seq | Buf
-------------------------------------------------------------------------------------------------------------------
_20459_              | 201      | sky130_fd_sc_hd__inv_2       | 3.75     | 2     | 0   | 1  
_20460_              | 201      | sky130_fd_sc_hd__inv_2       | 3.75     | 2     | 0   | 1  
_20461_              | 201      | sky130_fd_sc_hd__inv_2       | 3.75     | 2     | 0   | 1  
_20462_              | 116      | sky130_fd_sc_hd__clkinv_16   | 30.03    | 16    | 0   | 1  
_20463_              | 116      | sky130_fd_sc_hd__clkinv_16   | 30.03    | 16    | 0   | 1  
_20464_              | 203      | sky130_fd_sc_hd